[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/assoudi-typica-ai/magbert-ner-ar-tutorial/blob/main/MagBERT_NER_AR_Tutorial.ipynb)

# MagBERT-NER-AR — Quick Tutorial

[**TypicaAI/MagBERT-NER-AR**](https://huggingface.co/TypicaAI/MagBERT-NER-AR) is an Arabic (MSA) named entity recognition model fine-tuned from `asafaya/bert-base-arabic` on a corpus curated from **Moroccan newspapers**. It works on standard Arabic news text and is especially well adapted to Moroccan entities: names, cities and provinces, institutions, laws and date conventions.

This notebook shows how to:
1. load the model,
2. extract entities from Arabic text,
3. visualize the results on a few Moroccan-context examples.

*Runs on CPU or GPU.*

## 1. Install and load

In [ ]:
!pip install -q -U transformers spacy

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_ID = "TypicaAI/MagBERT-NER-AR"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID).to(device).eval()

entity_types = sorted({l.split("-", 1)[1] for l in model.config.id2label.values() if "-" in l})
print("Device:", device)
print("Entity types:", entity_types)

## 2. Quick start with the Hugging Face pipeline

The simplest way to run the model:

In [ ]:
from transformers import pipeline

ner = pipeline("token-classification", model=model, tokenizer=tokenizer,
               aggregation_strategy="first", device=0 if device == "cuda" else -1)

ner("يعقد بنك المغرب اجتماعه الفصلي بالرباط يوم الثلاثاء المقبل.")

The model uses the **BILUO** tagging scheme (`B`egin, `I`nside, `L`ast, `U`nit, `O`utside). The generic pipeline only understands `B-`/`I-`, so multi-word entities can come back split into pieces.

For clean entity spans, use the helper below.

## 3. Recommended: BILUO-aware entity extraction

In [ ]:
def extract_entities(text, max_length=512):
    """Return a list of entities: {type, text, score, start, end}."""
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True,
                    max_length=max_length, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    word_ids = enc.word_ids(0)
    with torch.no_grad():
        probs = model(**enc.to(device)).logits[0].softmax(-1).cpu()

    # 1) one label per word (first sub-token), span extended to the full word
    words, prev = [], None
    for i, wid in enumerate(word_ids):
        if wid is None:
            continue
        s, e = offsets[i]
        if wid != prev:
            score, idx = probs[i].max(-1)
            words.append([s, e, model.config.id2label[int(idx)], float(score)])
            prev = wid
        else:
            words[-1][1] = e

    # 2) group words into entities following BILUO
    entities, cur = [], None
    def close():
        nonlocal cur
        if cur:
            entities.append({"type": cur["type"], "text": text[cur["start"]:cur["end"]],
                             "score": round(sum(cur["scores"]) / len(cur["scores"]), 3),
                             "start": cur["start"], "end": cur["end"]})
            cur = None

    for s, e, tag, score in words:
        if tag in ("O", "PAD"):
            close(); continue
        prefix, _, etype = tag.partition("-")
        if prefix in ("B", "U") or cur is None or cur["type"] != etype:
            close()
            cur = {"type": etype, "start": s, "end": e, "scores": [score]}
        else:
            cur["end"] = e; cur["scores"].append(score)
        if prefix in ("L", "U"):
            close()
    close()
    return entities


pd.DataFrame(extract_entities("يعقد بنك المغرب اجتماعه الفصلي بالرباط يوم الثلاثاء المقبل."))

## 4. Visualization helper (right-to-left)

In [ ]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

_blank = spacy.blank("xx")

def show(text):
    ents = extract_entities(text)
    doc = _blank.make_doc(text)
    spans = [doc.char_span(e["start"], e["end"], label=e["type"], alignment_mode="expand") for e in ents]
    doc.ents = spacy.util.filter_spans([s for s in spans if s is not None])
    html = displacy.render(doc, style="ent", jupyter=False)
    display(HTML(f'<div dir="rtl" style="text-align:right;font-size:20px;line-height:2.3">{html}</div>'))
    if ents:
        display(pd.DataFrame(ents)[["type", "text", "score"]])

## 5. Examples — Moroccan news context

### Institutions & places

In [ ]:
show("استقبلت جامعة الأخوين في إفران وفدا من جامعة محمد الخامس يوم الاثنين.")

### Statistics

In [ ]:
show("أعلنت المندوبية السامية للتخطيط أن معدل البطالة بلغ حوالي 13 في المائة سنة 2024.")

### Local administration & people

In [ ]:
show("التقى رئيس جماعة شفشاون، كريم الإدريسي، بممثلي جمعيات المجتمع المدني.")

### Events, groups & Moroccan date format

In [ ]:
show("يحتفل المغاربة بذكرى المسيرة الخضراء في 6 نونبر من كل سنة.")

### Parliament & laws

In [ ]:
show("صادق مجلس النواب على مشروع قانون يتعلق بمدونة الأسرة.")

### Sport

In [ ]:
show("حل فريق المغرب الفاسي في المرتبة الثالثة بعد الجولة العاشرة من البطولة الوطنية.")

### Regions & tourism

In [ ]:
show("تعرف واحة سكورة في إقليم ورزازات إقبالا متزايدا من السياح الأوروبيين.")

### International

In [ ]:
show("زار وفد من الاتحاد الإفريقي مدينة العيون للاطلاع على مشاريع التنمية.")

## 6. Beyond Morocco: other Arab-country conventions

The model handles standard Arabic from any country. Here is the same sentence written with Moroccan, Egyptian and Levantine month names:

In [ ]:
for region, sent in {
    "Morocco": "انعقد المؤتمر يوم 15 دجنبر في الدار البيضاء بحضور ممثلين عن وزارة الصحة.",
    "Egypt":   "انعقد المؤتمر يوم 15 ديسمبر في القاهرة بحضور ممثلين عن وزارة الصحة.",
    "Levant":  "انعقد المؤتمر يوم 15 كانون الأول في بيروت بحضور ممثلين عن وزارة الصحة.",
}.items():
    print(f"── {region}")
    show(sent)

## 7. Process several texts at once

In [ ]:
texts = [sent for _, sent in examples]

rows = [{"sentence": i, **e} for i, t in enumerate(texts) for e in extract_entities(t)]
results = pd.DataFrame(rows)

print(f"{len(results)} entities found in {len(texts)} sentences\n")
print(results["type"].value_counts().to_string())
results

## 8. Try your own text

In [ ]:
text = "تنظم وزارة الثقافة مهرجان فاس للموسيقى العريقة بمشاركة فنانين من السنغال ومالي."  #@param {type:"string"}
show(text)

## Notes & limitations

* **Input:** Modern Standard Arabic in Arabic script. Darija, Arabizi and Latin-script text are out of scope.
* **Length:** the model was trained on sentences; split long documents into sentences before inference (max 512 sub-tokens).
* **Clitics:** prefixes such as *ب، ل، و* stay attached to the entity (e.g. *بالرباط*).
* **Rare types** (`LAW`, `ORDINAL`, `PRODUCT`, `LOC`) are less reliable than `PERSON`, `GPE`, `ORG`, `DATE`.
* **License:** see the [model card](https://huggingface.co/TypicaAI/MagBERT-NER-AR); released for education and research.

*Model by Hicham Assoudi — [Typica.ai](https://typica.ai). First trained April 2021, retrained August 2023.*